# Dataset cohort and buzz construction
This output-free notebook uses only the released derived tables. It reports the cohort, reconstructs one buzz's ramp/plateau/final phases from click times, and runs deterministic threshold/null diagnostics.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))
from dswp.buzz_thresholds import load_clicks, threshold_sweep, null_window_diagnostic, threshold_summary, segment_clicks, cohort_summary
DEPLOYMENTS = pd.read_csv(ROOT / 'data/dswp/deployment_summary.csv')
EVENTS = pd.read_csv(ROOT / 'data/dswp/buzz_events.csv')
CLICKS = load_clicks(ROOT / 'data/dswp/buzz_clicks.csv')
cohort_summary(DEPLOYMENTS, EVENTS)

The expected release cohort is **34 deployments, 20 individuals, 245 dives, and 3,686 buzzes**. Deployment, individual, and dive counts come from the deployment summary; the buzz count comes from the validated event table.

In [ ]:
event_id = CLICKS.event_id.iloc[0]
one = CLICKS.loc[CLICKS.event_id == event_id, 'click_time_from_buzz_start_s'].to_numpy()
segmentation = segment_clicks(one, threshold=.7)
segmentation

The segmentation above is a deterministic diagnostic from released click times. The reviewed acoustic boundaries in `buzz_events.csv` remain the production reference; the diagnostic is not evidence that a single threshold uniquely defines a buzz phase.

In [ ]:
sweep = threshold_sweep(CLICKS)
nulls = null_window_diagnostic(CLICKS, n_windows=20, seed=42)
diagnostics = threshold_summary(sweep, nulls)
diagnostics

The sweep reproduces the three SI diagnostics over alpha = 0.05--0.95: plateau ICI CV, its numerical derivative, and median null-window CV minus plateau CV. The production setting is alpha = 0.7, with a symmetric 0.3 s rate window, 0.05 s step, and five-point centered smoothing.

Known issue touchpoint: threshold-selection wording. The alpha = 0.7 plateau CV is 0.1198, matching the reported 0.12, but the deterministic 20-window null calculation peaks at alpha = 0.55 rather than 0.7; the values are nearly flat from 0.55 to 0.70. The SI should not describe 0.7 as a unique numerical optimum without specifying the randomization and selection rule.

Known issue touchpoint: upstream buzz construction. Released click times are sufficient to reproduce phase segmentation and threshold sensitivity, but not click detection from raw audio or the full-record dive panel.

All tables remain in memory. No notebook outputs, figures, or derived files are written.